# VeReMi Extension: Leakage-Safe EDA and Preprocessing Audit

This notebook performs reproducible exploratory data analysis on the **VeReMi Extension `_1416` archives** without manually extracting their nested ZIP files.

Important rules implemented here:

- `A0` identifies a benign vehicle.
- Every nonzero code (`A1`, `A2`, `A13`, `A17`, etc.) identifies an attacking vehicle.
- Received type-3 messages are labelled from the **sender's** attack code within the same simulation run.
- Simulator ground truth is used only for audit diagrams. Ground-truth-derived columns are forbidden as deployable model inputs.
- Random message-level train/test splitting is prohibited because the same mobility runs are reused across attack scenarios.


## Kaggle setup

1. Create a Kaggle Dataset containing the 19 `_1416.zip` files and `README.md`.
2. Attach that dataset to this notebook with **Add Input**.
3. Run all cells. The notebook auto-detects the directory containing the ZIP files.
4. Download the generated results from `/kaggle/working/veremi_eda_outputs`.

For a quick run, keep `MAX_INNER_PER_ARCHIVE = 1`. For the full two-hour audit, set it to `0`.


In [ ]:
from pathlib import Path
import collections
import csv
import hashlib
import io
import json
import math
import random
import re
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'axes.titleweight': 'bold'})

SEED = 20261021
MAX_INNER_PER_ARCHIVE = 1   # 1 = fast EDA; 0 = all nested runs
SAMPLE_PER_CLASS_PER_SCENARIO = 5000
OUTPUT_DIR = Path('/kaggle/working/veremi_eda_outputs')
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', OUTPUT_DIR)


In [ ]:
# Locate the attached Kaggle dataset automatically.
candidate_dirs = []
for directory in Path('/kaggle/input').rglob('*'):
    if directory.is_dir() and len(list(directory.glob('*_1416.zip'))) >= 10:
        candidate_dirs.append(directory)

if not candidate_dirs:
    raise FileNotFoundError('No directory containing *_1416.zip was found under /kaggle/input. Attach the dataset first.')

DATASET_DIR = sorted(candidate_dirs, key=lambda p: len(list(p.glob('*_1416.zip'))), reverse=True)[0]
archives = sorted(DATASET_DIR.glob('*_1416.zip'))
print('Dataset directory:', DATASET_DIR)
print('Archives found:', len(archives))
assert len(archives) == 19, f'Expected 19 archives, found {len(archives)}'

inventory = pd.DataFrame({
    'archive': [p.name for p in archives],
    'size_mb': [p.stat().st_size / 1024**2 for p in archives],
})
display(inventory)


In [ ]:
# Dataset archive-size diagram
plt.figure(figsize=(11, 7))
ordered = inventory.sort_values('size_mb', ascending=True)
sns.barplot(data=ordered, y='archive', x='size_mb', color='#386cb0')
plt.title('Compressed size of VeReMi Extension scenarios')
plt.xlabel('Archive size (MiB)')
plt.ylabel('')
plt.tight_layout()
plt.savefig(FIGURE_DIR / '01_archive_sizes.png', bbox_inches='tight')
plt.show()


In [ ]:
TRACE_RE = re.compile(r'traceJSON-(\d+)-(\d+)-A(\d+)-')
SCENARIO_RE = re.compile(r'^(?P<attack>.+)_(?P<period>\d{4})\.zip$')

def vector2(value):
    if not isinstance(value, list) or len(value) < 2:
        return np.nan, np.nan
    try:
        return float(value[0]), float(value[1])
    except (TypeError, ValueError):
        return np.nan, np.nan

def norm2(x, y):
    return math.hypot(x, y) if np.isfinite(x) and np.isfinite(y) else np.nan

def distance(a, b):
    ax, ay = vector2(a); bx, by = vector2(b)
    return math.hypot(ax-bx, ay-by) if all(np.isfinite([ax, ay, bx, by])) else np.nan

def iter_json_lines(stream):
    with io.TextIOWrapper(stream, encoding='utf-8', errors='replace') as reader:
        for line_number, line in enumerate(reader, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSON line {line_number}') from exc
            if isinstance(obj, dict):
                yield obj

class Reservoir:
    def __init__(self, capacity, seed):
        self.capacity = capacity
        self.rng = random.Random(seed)
        self.seen = 0
        self.rows = []
    def add(self, row):
        self.seen += 1
        if len(self.rows) < self.capacity:
            self.rows.append(row)
        else:
            index = self.rng.randrange(self.seen)
            if index < self.capacity:
                self.rows[index] = row

def attack_mapping(inner):
    mapping = {}
    attack_codes = {}
    for entry in inner.infolist():
        match = TRACE_RE.search(Path(entry.filename).name)
        if match:
            vehicle_id, _, attack_code = map(int, match.groups())
            mapping[vehicle_id] = int(attack_code != 0)
            attack_codes[vehicle_id] = attack_code
    return mapping, attack_codes

def load_truth(inner):
    entries = [e for e in inner.infolist() if 'traceGroundTruthJSON-' in e.filename and e.filename.endswith('.json')]
    if not entries:
        return {}
    assert len(entries) == 1, f'Unexpected ground-truth file count: {len(entries)}'
    truth = {}
    with inner.open(entries[0]) as stream:
        for obj in iter_json_lines(stream):
            if isinstance(obj.get('messageID'), int):
                truth[obj['messageID']] = obj
    return truth

print('Parser ready.')


In [ ]:
def make_row(obj, receiver_id, attacker_map, attack_codes, truth, scenario, period, run_id):
    if obj.get('type') != 3 or not isinstance(obj.get('sender'), int) or not isinstance(obj.get('messageID'), int):
        return None
    sender, message_id = obj['sender'], obj['messageID']
    px, py = vector2(obj.get('pos')); pnx, pny = vector2(obj.get('pos_noise'))
    sx, sy = vector2(obj.get('spd')); snx, sny = vector2(obj.get('spd_noise'))
    ax, ay = vector2(obj.get('acl')); anx, any_ = vector2(obj.get('acl_noise'))
    hx, hy = vector2(obj.get('hed')); hnx, hny = vector2(obj.get('hed_noise'))
    receive_time, send_time = obj.get('rcvTime', np.nan), obj.get('sendTime', np.nan)
    try: latency = float(receive_time) - float(send_time)
    except (TypeError, ValueError): latency = np.nan
    is_attack = int(attacker_map.get(sender, 0) == 1)
    gt = truth.get(message_id)
    return {
        'scenario': scenario, 'period': period, 'run_id': run_id,
        'receiver_id': receiver_id, 'sender_id': sender,
        'sender_pseudo': obj.get('senderPseudo'), 'message_id': message_id,
        'receive_time': receive_time, 'send_time': send_time, 'latency': latency,
        'pos_x': px, 'pos_y': py, 'pos_noise_x': pnx, 'pos_noise_y': pny,
        'speed_x': sx, 'speed_y': sy, 'speed_norm': norm2(sx, sy),
        'speed_noise_x': snx, 'speed_noise_y': sny,
        'accel_x': ax, 'accel_y': ay, 'accel_norm': norm2(ax, ay),
        'accel_noise_x': anx, 'accel_noise_y': any_,
        'heading_x': hx, 'heading_y': hy, 'heading_norm': norm2(hx, hy),
        'heading_noise_x': hnx, 'heading_noise_y': hny,
        'attack_code': attack_codes.get(sender, 0),
        'label_binary': is_attack,
        'label_attack': scenario if is_attack else 'Benign',
        # Audit-only: simulator truth is unavailable in deployment.
        'audit_truth_available': int(gt is not None),
        'audit_position_error': distance(obj.get('pos'), gt.get('pos')) if gt else np.nan,
        'audit_speed_error': distance(obj.get('spd'), gt.get('spd')) if gt else np.nan,
        'audit_accel_error': distance(obj.get('acl'), gt.get('acl')) if gt else np.nan,
        'audit_heading_error': distance(obj.get('hed'), gt.get('hed')) if gt else np.nan,
    }

def audit_archive(path, max_inner=1, sample_per_class=5000):
    match = SCENARIO_RE.match(path.name)
    scenario, period = match.group('attack'), match.group('period')
    reservoirs = {0: Reservoir(sample_per_class, SEED+1), 1: Reservoir(sample_per_class, SEED+2)}
    summary = collections.Counter()
    senders, receivers = set(), set()
    run_rows = []
    with zipfile.ZipFile(path) as outer:
        inner_entries = sorted([e for e in outer.infolist() if e.filename.lower().endswith('.zip')],
                               key=lambda e: e.file_size, reverse=True)
        if max_inner:
            inner_entries = inner_entries[:max_inner]
        for nested in inner_entries:
            raw = outer.read(nested)
            run_id = Path(nested.filename).stem
            with zipfile.ZipFile(io.BytesIO(raw)) as inner:
                attacker_map, attack_codes = attack_mapping(inner)
                truth = load_truth(inner)
                traces = [e for e in inner.infolist() if 'traceJSON-' in e.filename and
                          'traceGroundTruthJSON-' not in e.filename and e.filename.endswith('.json')]
                local = collections.Counter()
                for entry in traces:
                    fm = TRACE_RE.search(Path(entry.filename).name)
                    if not fm: continue
                    receiver_id = int(fm.group(1))
                    with inner.open(entry) as stream:
                        for obj in iter_json_lines(stream):
                            row = make_row(obj, receiver_id, attacker_map, attack_codes, truth,
                                           scenario, period, run_id)
                            if row is None: continue
                            label = row['label_binary']
                            reservoirs[label].add(row)
                            summary['rows'] += 1; local['rows'] += 1
                            summary['malicious' if label else 'benign'] += 1
                            local['malicious' if label else 'benign'] += 1
                            summary['truth_missing'] += int(not row['audit_truth_available'])
                            senders.add(row['sender_id']); receivers.add(row['receiver_id'])
                run_rows.append({'scenario': scenario, 'run_id': run_id,
                                 'rows': local['rows'], 'benign': local['benign'],
                                 'malicious': local['malicious'],
                                 'attacker_vehicles': sum(attacker_map.values()),
                                 'vehicles': len(attacker_map), 'ground_truth_messages': len(truth)})
    result = {'scenario': scenario, 'period': period, 'rows': summary['rows'],
              'benign': summary['benign'], 'malicious': summary['malicious'],
              'malicious_percent': 100*summary['malicious']/summary['rows'] if summary['rows'] else np.nan,
              'unique_senders': len(senders), 'unique_receivers': len(receivers),
              'truth_missing': summary['truth_missing']}
    return result, reservoirs[0].rows + reservoirs[1].rows, run_rows


In [ ]:
# Stream the nested archives and create a balanced EDA sample.
scenario_summaries, sample_rows, run_manifest = [], [], []
for number, archive in enumerate(archives, 1):
    print(f'[{number:02d}/{len(archives)}] {archive.name}', flush=True)
    summary, sample, runs = audit_archive(archive, MAX_INNER_PER_ARCHIVE,
                                          SAMPLE_PER_CLASS_PER_SCENARIO)
    scenario_summaries.append(summary)
    sample_rows.extend(sample)
    run_manifest.extend(runs)

summary_df = pd.DataFrame(scenario_summaries).sort_values('scenario').reset_index(drop=True)
sample_df = pd.DataFrame(sample_rows)
run_df = pd.DataFrame(run_manifest)

summary_df.to_csv(OUTPUT_DIR / 'scenario_summary.csv', index=False)
run_df.to_csv(OUTPUT_DIR / 'run_manifest.csv', index=False)
sample_df.to_csv(OUTPUT_DIR / 'balanced_eda_sample.csv.gz', index=False, compression='gzip')
print(f'Processed {summary_df.rows.sum():,} received messages; saved {len(sample_df):,} sampled rows.')
display(summary_df)


In [ ]:
# Diagram 2: binary class distribution by scenario
plot_df = summary_df.melt(id_vars='scenario', value_vars=['benign', 'malicious'],
                          var_name='class', value_name='messages')
plt.figure(figsize=(13, 8))
sns.barplot(data=plot_df, y='scenario', x='messages', hue='class',
            palette={'benign':'#4daf4a', 'malicious':'#e41a1c'})
plt.title('Benign and malicious received messages by scenario')
plt.xlabel('Received-message observations'); plt.ylabel('')
plt.legend(title='Class')
plt.tight_layout(); plt.savefig(FIGURE_DIR/'02_class_distribution_by_scenario.png', bbox_inches='tight'); plt.show()

# Diagram 3: malicious percentage
plt.figure(figsize=(12, 7))
sns.barplot(data=summary_df.sort_values('malicious_percent'), y='scenario', x='malicious_percent', color='#e6550d')
plt.axvline(summary_df.malicious.sum()/summary_df.rows.sum()*100, color='black', ls='--', label='Overall rate')
plt.title('Malicious-observation percentage by scenario'); plt.xlabel('Malicious observations (%)'); plt.ylabel('')
plt.legend(); plt.tight_layout(); plt.savefig(FIGURE_DIR/'03_malicious_percentage.png', bbox_inches='tight'); plt.show()


In [ ]:
# Diagram 4: missing-value profile in the balanced sample
missing = (sample_df.isna().mean()*100).sort_values(ascending=False)
missing = missing[missing > 0].reset_index()
missing.columns = ['feature', 'missing_percent']
display(missing)
if not missing.empty:
    plt.figure(figsize=(10, max(4, len(missing)*0.3)))
    sns.barplot(data=missing, y='feature', x='missing_percent', color='#756bb1')
    plt.title('Missing values in balanced EDA sample'); plt.xlabel('Missing (%)'); plt.ylabel('')
    plt.tight_layout(); plt.savefig(FIGURE_DIR/'04_missing_values.png', bbox_inches='tight'); plt.show()


In [ ]:
# Diagram 5: deployable numerical feature distributions
features = ['speed_norm', 'accel_norm', 'latency', 'pos_noise_x', 'speed_noise_x']
fig, axes = plt.subplots(len(features), 1, figsize=(11, 3.2*len(features)))
for ax, feature in zip(axes, features):
    clean = sample_df[[feature, 'label_binary']].replace([np.inf, -np.inf], np.nan).dropna()
    # Clip only for visualization; raw values remain unchanged.
    lo, hi = clean[feature].quantile([0.01, 0.99])
    clean = clean[clean[feature].between(lo, hi)]
    sns.histplot(data=clean, x=feature, hue='label_binary', stat='density', common_norm=False,
                 element='step', bins=60, ax=ax, palette={0:'#4daf4a',1:'#e41a1c'})
    ax.set_title(f'{feature}: benign versus malicious (1st–99th percentile shown)')
plt.tight_layout(); plt.savefig(FIGURE_DIR/'05_feature_distributions.png', bbox_inches='tight'); plt.show()


In [ ]:
# Diagram 6: simulator-truth discrepancy—EDA only, never a model input
audit_features = ['audit_position_error','audit_speed_error','audit_accel_error','audit_heading_error']
audit_long = sample_df.melt(id_vars=['label_binary'], value_vars=audit_features,
                            var_name='audit_metric', value_name='error').dropna()
audit_long['log1p_error'] = np.log1p(audit_long['error'].clip(lower=0))
plt.figure(figsize=(12, 6))
sns.boxplot(data=audit_long, x='audit_metric', y='log1p_error', hue='label_binary',
            showfliers=False, palette={0:'#4daf4a',1:'#e41a1c'})
plt.xticks(rotation=20, ha='right'); plt.ylabel('log(1 + simulator-truth error)')
plt.title('Attack effect confirmed using simulator truth (audit only)')
plt.tight_layout(); plt.savefig(FIGURE_DIR/'06_ground_truth_audit.png', bbox_inches='tight'); plt.show()


In [ ]:
# Diagram 7: spatial pattern sample
spatial = sample_df[['pos_x','pos_y','label_binary','scenario']].dropna().sample(
    min(30000, len(sample_df)), random_state=SEED)
plt.figure(figsize=(10, 8))
sns.scatterplot(data=spatial, x='pos_x', y='pos_y', hue='label_binary',
                palette={0:'#377eb8',1:'#e41a1c'}, alpha=.25, s=10)
plt.title('Reported V2X positions in the balanced EDA sample')
plt.xlabel('Position X'); plt.ylabel('Position Y'); plt.axis('equal')
plt.tight_layout(); plt.savefig(FIGURE_DIR/'07_position_scatter.png', bbox_inches='tight'); plt.show()


In [ ]:
# Diagram 8: correlation heatmap for allowed raw/model-derived numerical variables
forbidden = [c for c in sample_df.columns if c.startswith('audit_')]
identifier_columns = ['receiver_id','sender_id','sender_pseudo','message_id','attack_code','label_binary']
numeric = sample_df.select_dtypes(include='number').drop(columns=forbidden, errors='ignore')
numeric = numeric.drop(columns=[c for c in identifier_columns if c in numeric], errors='ignore')
corr = numeric.corr(method='spearman')
plt.figure(figsize=(15, 12))
sns.heatmap(corr, cmap='vlag', center=0, vmin=-1, vmax=1, square=False)
plt.title('Spearman correlation of deployable numerical features')
plt.tight_layout(); plt.savefig(FIGURE_DIR/'08_feature_correlation.png', bbox_inches='tight'); plt.show()


In [ ]:
# Leakage-safe preprocessing contract
AUDIT_ONLY = [c for c in sample_df if c.startswith('audit_')]
IDENTIFIERS = ['scenario','period','run_id','receiver_id','sender_id','sender_pseudo',
               'message_id','attack_code','receive_time','send_time']
TARGETS = ['label_binary','label_attack']
MODEL_FEATURES = [c for c in sample_df.columns if c not in AUDIT_ONLY + IDENTIFIERS + TARGETS]

contract = {
    'model_features': MODEL_FEATURES,
    'targets': TARGETS,
    'group_split_key': 'run_id / shared temporal block',
    'forbidden_model_inputs': AUDIT_ONLY,
    'identifier_columns': IDENTIFIERS,
    'rules': [
        'Fit imputers/scalers/feature selection on training runs only.',
        'Create temporal windows only after assigning whole runs to partitions.',
        'Do not use SMOTE on time-series windows before splitting.',
        'Do not randomly split messages; mobility traces recur across scenarios.',
        'Use macro-F1, per-class recall, MCC and AUPRC in addition to accuracy.'
    ]
}
(OUTPUT_DIR/'preprocessing_contract.json').write_text(json.dumps(contract, indent=2))
display(pd.DataFrame({'model_feature': MODEL_FEATURES}))
print('Forbidden audit-only columns:', AUDIT_ONLY)


In [ ]:
# Time-block audit: determine whether a defensible 3-way split is possible.
run_df['time_block'] = run_df['run_id'].str.extract(r'VeReMi_(\d+_\d+)_', expand=False).str.replace('_','-')
block_summary = run_df.groupby('time_block', dropna=False).agg(
    runs=('run_id','size'), rows=('rows','sum'), malicious=('malicious','sum'),
    empty_runs=('rows', lambda x: int((x==0).sum()))).reset_index()
display(block_summary)
block_summary.to_csv(OUTPUT_DIR/'time_block_summary.csv', index=False)

usable_blocks = block_summary.loc[block_summary.rows > 0, 'time_block'].nunique()
if usable_blocks < 3:
    print('WARNING: fewer than three non-empty shared time blocks are available.')
    print('Use _0709 for an independent density/domain-shift test and reserve MixAll_0024 for final stress testing.')


In [ ]:
# Final output inventory
print('Tables:')
for path in sorted(OUTPUT_DIR.glob('*')):
    if path.is_file(): print(' ', path.name, f'{path.stat().st_size/1024:.1f} KiB')
print('Figures:')
for path in sorted(FIGURE_DIR.glob('*.png')):
    print(' ', path.name)
print('\nEDA completed successfully.')
